# BeetleCast 07 — Ground-truth year audit

This notebook provides a reproducible answer to:

> **Which years have actual supplied bark-beetle damage polygons?**

It audits the original label file rather than inferring truth from Sentinel-2, AlphaEarth, LULC, or model predictions.

The notebook:

1. locates the original bark-beetle locations file and the F3 AOI;
2. inspects all label attributes for year values;
3. extracts the survey/source year from `tile_name`;
4. explicitly checks for 2024 and 2025 labels;
5. clips the labels to the F3 AOI;
6. exports a year summary, an audit report, and cleaned AOI labels.

Expected conclusion for the supplied dataset:

- actual polygon labels exist for **2022 and 2023**;
- no independent polygon labels exist for **2024 or 2025**;
- 2024 and 2025 outputs are therefore predictions, not validated ground truth.


In [8]:
from pathlib import Path
import json
import re

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

cwd = Path.cwd()

if (cwd / "hackathon_data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "hackathon_data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "ground_truth_audit"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


PROJECT_ROOT: /Users/hemat/Desktop/hackathon-demo
OUTPUT_ROOT: /Users/hemat/Desktop/hackathon-demo/outputs/ground_truth_audit


## 1. Locate the original label and AOI files

In [9]:
def score_label_candidate(path):
    name = path.name.lower()
    score = 0

    if "beetle" in name or "bettle" in name:
        score += 4
    if "location" in name:
        score += 4
    if "label" in name:
        score += 2
    if "aoi" in name:
        score -= 10
    if "processed" in str(path).lower():
        score -= 2

    return score

def score_aoi_candidate(path):
    name = path.name.lower()
    score = 0

    if "aoi" in name:
        score += 8
    if "barkbeetle" in name or "bark_beetle" in name:
        score += 2
    if "location" in name or "label" in name:
        score -= 8

    return score

search_roots = [
    PROJECT_ROOT,
    PROJECT_ROOT / "hackathon_data",
]

vector_paths = []
for root in search_roots:
    if root.exists():
        vector_paths.extend(root.rglob("*.geojson"))
        vector_paths.extend(root.rglob("*.gpkg"))
        vector_paths.extend(root.rglob("*.shp"))

# Deduplicate resolved paths.
vector_paths = sorted(set(path.resolve() for path in vector_paths))

label_candidates = sorted(
    vector_paths,
    key=score_label_candidate,
    reverse=True,
)

aoi_candidates = sorted(
    vector_paths,
    key=score_aoi_candidate,
    reverse=True,
)

LABEL_PATH = next(
    (
        path for path in label_candidates
        if score_label_candidate(path) > 0
    ),
    None,
)

AOI_PATH = next(
    (
        path for path in aoi_candidates
        if score_aoi_candidate(path) > 0
    ),
    None,
)

assert LABEL_PATH is not None, (
    "Could not find the original bark-beetle locations file. "
    "Expected a file similar to F3_bark_beetle_locations.geojson."
)

assert AOI_PATH is not None, (
    "Could not find the F3 AOI file. "
    "Expected a file similar to F3_Germany_BarkBeetle_aoi.geojson."
)

print("Label file:", LABEL_PATH)
print("AOI file:", AOI_PATH)


Label file: /Users/hemat/Desktop/hackathon-demo/hackathon_data/raw/F3_bark_beetle_locations.geojson
AOI file: /Users/hemat/Desktop/hackathon-demo/hackathon_demo_data/boundary/D_Chablis_Vineyard_aoi.geojson


## 2. Inspect the raw files

In [10]:
labels_raw = gpd.read_file(LABEL_PATH)
aoi = gpd.read_file(AOI_PATH)

print("RAW LABEL FILE")
print("--------------")
print("Rows:", len(labels_raw))
print("CRS:", labels_raw.crs)
print("Geometry types:")
print(labels_raw.geom_type.value_counts())
print("Columns:", labels_raw.columns.tolist())

print()
print("AOI FILE")
print("--------")
print("Rows:", len(aoi))
print("CRS:", aoi.crs)
print("Geometry types:")
print(aoi.geom_type.value_counts())
print("Columns:", aoi.columns.tolist())

assert "geometry" in labels_raw.columns
assert not labels_raw.empty
assert not aoi.empty


RAW LABEL FILE
--------------
Rows: 14358
CRS: EPSG:4326
Geometry types:
MultiPolygon    14358
Name: count, dtype: int64
Columns: ['fid', 'tile_name', 'area_ha', 'geometry']

AOI FILE
--------
Rows: 1
CRS: EPSG:4326
Geometry types:
Polygon    1
Name: count, dtype: int64
Columns: ['geometry']


## 3. Search every non-geometry attribute for four-digit years

This prevents the audit from relying only on one assumed column.


In [11]:
year_occurrences = []

for column in labels_raw.columns:
    if column == "geometry":
        continue

    values = labels_raw[column].astype(str)

    extracted = values.str.extractall(r"(?<!\d)(20\d{2})(?!\d)")

    if extracted.empty:
        continue

    counts = extracted[0].value_counts().sort_index()

    for year_text, count in counts.items():
        year_occurrences.append({
            "column": column,
            "year": int(year_text),
            "occurrences": int(count),
        })

attribute_years = pd.DataFrame(year_occurrences)

if attribute_years.empty:
    print("No four-digit years found in any attribute.")
else:
    display(attribute_years)

all_attribute_years = sorted(
    attribute_years["year"].unique().tolist()
) if not attribute_years.empty else []

print("All years found anywhere in label attributes:", all_attribute_years)


,column,year,occurrences
0,fid,2000,1
1,fid,2001,1
2,fid,2002,1
3,fid,2003,1
4,fid,2004,1
...,...,...,...
97,fid,2097,1
98,fid,2098,1
99,fid,2099,1
100,tile_name,2022,2413


All years found anywhere in label attributes: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059, 2060, 2061, 2062, 2063, 2064, 2065, 2066, 2067, 2068, 2069, 2070, 2071, 2072, 2073, 2074, 2075, 2076, 2077, 2078, 2079, 2080, 2081, 2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099]


## 4. Extract the label year from `tile_name`

The supplied year appears in names such as:

```text
..._rp_2022_...
..._rp_2023_...
```

A strict pattern is used so unrelated numbers in the filename are not mistaken for years.


In [12]:
assert "tile_name" in labels_raw.columns, (
    "The supplied label file does not contain tile_name."
)

strict_year = labels_raw["tile_name"].astype(str).str.extract(
    r"_rp_(20\d{2})_",
    expand=False,
)

labels_raw = labels_raw.copy()
labels_raw["label_year"] = pd.to_numeric(
    strict_year,
    errors="coerce",
).astype("Int64")

unmatched = int(labels_raw["label_year"].isna().sum())

print("Unmatched tile_name rows:", unmatched)
assert unmatched == 0, (
    "Some labels do not contain the expected _rp_YEAR_ pattern."
)

raw_year_summary = (
    labels_raw.groupby("label_year", dropna=False)
    .agg(
        polygon_count=("geometry", "size"),
        labelled_area_ha=("area_ha", "sum")
        if "area_ha" in labels_raw.columns
        else ("geometry", "size"),
    )
    .reset_index()
)

display(raw_year_summary)

raw_years = sorted(
    labels_raw["label_year"].dropna().astype(int).unique().tolist()
)

print("Unique strict label years:", raw_years)


Unmatched tile_name rows: 0


,label_year,polygon_count,labelled_area_ha
0,2022,2413,384.502632
1,2023,11945,1588.047311


Unique strict label years: [2022, 2023]


## 5. Explicitly test the 2022–2025 years

In [13]:
audit_rows = []

for year in range(2022, 2026):
    count = int(
        (labels_raw["label_year"] == year).sum()
    )

    audit_rows.append({
        "year": year,
        "raw_ground_truth_polygon_count": count,
        "has_supplied_ground_truth": count > 0,
        "status": (
            "labelled ground truth"
            if count > 0
            else "no supplied polygon ground truth"
        ),
    })

year_audit = pd.DataFrame(audit_rows)
display(year_audit)

assert int(
    year_audit.loc[
        year_audit["year"] == 2022,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) > 0

assert int(
    year_audit.loc[
        year_audit["year"] == 2023,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) > 0

assert int(
    year_audit.loc[
        year_audit["year"] == 2024,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) == 0

assert int(
    year_audit.loc[
        year_audit["year"] == 2025,
        "raw_ground_truth_polygon_count",
    ].iloc[0]
) == 0

print("Assertions passed:")
print("- 2022 labels exist")
print("- 2023 labels exist")
print("- 2024 labels do not exist")
print("- 2025 labels do not exist")


,year,raw_ground_truth_polygon_count,has_supplied_ground_truth,status
0,2022,2413,True,labelled ground truth
1,2023,11945,True,labelled ground truth
2,2024,0,False,no supplied polygon ground truth
3,2025,0,False,no supplied polygon ground truth


Assertions passed:
- 2022 labels exist
- 2023 labels exist
- 2024 labels do not exist
- 2025 labels do not exist


## 6. Clip the ground truth to the F3 AOI

In [14]:
if labels_raw.crs is None:
    raise ValueError("The label file has no CRS.")

if aoi.crs is None:
    raise ValueError("The AOI file has no CRS.")

aoi_in_label_crs = aoi.to_crs(labels_raw.crs)

# Repair invalid geometries conservatively before clipping.
labels_valid = labels_raw[
    labels_raw.geometry.notna()
    & ~labels_raw.geometry.is_empty
].copy()

labels_valid["geometry"] = labels_valid.geometry.make_valid()

aoi_geometry = aoi_in_label_crs.geometry.union_all()

labels_in_aoi = labels_valid[
    labels_valid.geometry.intersects(aoi_geometry)
].copy()

labels_in_aoi["geometry"] = labels_in_aoi.geometry.intersection(
    aoi_geometry
)

labels_in_aoi = labels_in_aoi[
    labels_in_aoi.geometry.notna()
    & ~labels_in_aoi.geometry.is_empty
].copy()

# Calculate area in a suitable projected CRS.
projected_crs = aoi_in_label_crs.estimate_utm_crs()
labels_projected = labels_in_aoi.to_crs(projected_crs)
labels_in_aoi["clipped_area_ha"] = (
    labels_projected.geometry.area / 10_000
)

aoi_year_summary = (
    labels_in_aoi.groupby("label_year")
    .agg(
        polygon_count=("geometry", "size"),
        clipped_area_ha=("clipped_area_ha", "sum"),
    )
    .reset_index()
)

display(aoi_year_summary)

aoi_years = sorted(
    labels_in_aoi["label_year"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print("Ground-truth years inside F3 AOI:", aoi_years)

assert aoi_years == [2022, 2023], (
    f"Unexpected AOI label years: {aoi_years}"
)


,label_year,polygon_count,clipped_area_ha


Ground-truth years inside F3 AOI: []


AssertionError: Unexpected AOI label years: []

## 7. Visual evidence

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    constrained_layout=True,
)

raw_plot = raw_year_summary.copy()
raw_plot["label_year"] = raw_plot["label_year"].astype(str)

axes[0].bar(
    raw_plot["label_year"],
    raw_plot["polygon_count"],
)
axes[0].set_title("Raw supplied damage polygons by year")
axes[0].set_xlabel("Label year")
axes[0].set_ylabel("Polygon count")

aoi_plot = aoi_year_summary.copy()
aoi_plot["label_year"] = aoi_plot["label_year"].astype(str)

axes[1].bar(
    aoi_plot["label_year"],
    aoi_plot["polygon_count"],
)
axes[1].set_title("Damage polygons inside the F3 AOI")
axes[1].set_xlabel("Label year")
axes[1].set_ylabel("Polygon count")

figure_path = OUTPUT_ROOT / "ground_truth_year_counts.png"
plt.savefig(
    figure_path,
    dpi=220,
    bbox_inches="tight",
)
plt.show()

print("Saved:", figure_path)


## 8. Export the reproducible audit outputs

In [ ]:
raw_summary_path = OUTPUT_ROOT / "raw_ground_truth_year_summary.csv"
aoi_summary_path = OUTPUT_ROOT / "aoi_ground_truth_year_summary.csv"
audit_path = OUTPUT_ROOT / "ground_truth_availability_2022_2025.csv"
attribute_years_path = OUTPUT_ROOT / "all_attribute_year_occurrences.csv"
clean_labels_path = OUTPUT_ROOT / "F3_ground_truth_2022_2023.gpkg"

raw_year_summary.to_csv(raw_summary_path, index=False)
aoi_year_summary.to_csv(aoi_summary_path, index=False)
year_audit.to_csv(audit_path, index=False)
attribute_years.to_csv(attribute_years_path, index=False)

labels_in_aoi.to_file(
    clean_labels_path,
    layer="bark_beetle_ground_truth",
    driver="GPKG",
)

report = {
    "source_label_file": str(LABEL_PATH),
    "source_aoi_file": str(AOI_PATH),
    "raw_polygon_count": int(len(labels_raw)),
    "raw_label_years": raw_years,
    "aoi_polygon_count": int(len(labels_in_aoi)),
    "aoi_label_years": aoi_years,
    "has_2022_ground_truth": 2022 in raw_years,
    "has_2023_ground_truth": 2023 in raw_years,
    "has_2024_ground_truth": 2024 in raw_years,
    "has_2025_ground_truth": 2025 in raw_years,
    "conclusion": (
        "The supplied polygon ground truth covers 2022 and 2023 only. "
        "No supplied polygon ground truth is available for 2024 or 2025."
    ),
}

report_path = OUTPUT_ROOT / "ground_truth_audit_report.json"
with open(report_path, "w") as handle:
    json.dump(report, handle, indent=2)

print("Saved:")
for path in [
    raw_summary_path,
    aoi_summary_path,
    audit_path,
    attribute_years_path,
    clean_labels_path,
    report_path,
]:
    print("-", path)


## 9. Final conclusion

In [ ]:
print("GROUND-TRUTH AUDIT COMPLETE")
print("---------------------------")
print(
    f"Raw supplied polygons: {len(labels_raw):,}"
)
print(
    "Raw supplied label years:",
    raw_years,
)
print(
    f"Polygons inside F3 AOI: {len(labels_in_aoi):,}"
)
print(
    "Label years inside F3 AOI:",
    aoi_years,
)
print()
print("Ground-truth availability:")
for row in year_audit.itertuples(index=False):
    print(
        f"- {row.year}: "
        f"{row.raw_ground_truth_polygon_count:,} polygons — "
        f"{row.status}"
    )
print()
print(
    "Conclusion: only 2022 and 2023 have supplied "
    "bark-beetle polygon ground truth."
)
print(
    "Any 2024 or 2025 risk map must be described as a "
    "prediction or inspection-priority map unless new "
    "independent labels are added."
)
